<style>
/* Minimal fixes only: keep the existing site design, but make notebook code and wide tables usable. */
.highlight pre,
div.highlight,
div.input_area,
div.output_area pre {
  background: #1f1f1f !important;
  color: #f8f8f2 !important;
  border-color: #444 !important;
}
.highlight span { color: inherit !important; }
.rendered_html,
.jp-RenderedHTMLCommon {
  max-width: 100%;
  overflow-x: auto;
  -webkit-overflow-scrolling: touch;
}
.rendered_html table,
.jp-RenderedHTMLCommon table {
  display: block;
  width: max-content;
  max-width: 100%;
  min-width: 680px;
  overflow-x: auto;
  -webkit-overflow-scrolling: touch;
}
.rendered_html td,
.rendered_html th,
.jp-RenderedHTMLCommon td,
.jp-RenderedHTMLCommon th {
  white-space: nowrap;
}
</style>

The dataset behind this article was generated from a resumable local SQLite checkpoint of the public 2026 CrossFit Open leaderboard API. I exported the normalized tables into a compact GitHub Release asset, excluded raw API response bodies, converted height and weight to metric units, and use the release download below as the source for the rest of the analysis.

The practical question I care about is percentile cutoffs: given a sex and age-group division, what rank and score did an athlete need for the 50th, 75th, 90th, and 95th percentiles?


## Load the release asset

The tarball is published as a GitHub Release asset rather than tracked in the repository. The notebook downloads it, verifies the SHA-256, and then loads the CSVs from the extracted archive.

Release asset: <https://github.com/leblancfg/leblancfg.github.io/releases/download/crossfit-open-2026-compact-v1/crossfit-open-2026-compact-v1.tar.gz>. Checksum file: <https://github.com/leblancfg/leblancfg.github.io/releases/download/crossfit-open-2026-compact-v1/SHA256SUMS>.


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import tarfile
import urllib.request
from collections import defaultdict
from pathlib import Path

from IPython.display import Markdown, display

RELEASE_TAG = "crossfit-open-2026-compact-v1"
ASSET_NAME = "crossfit-open-2026-compact-v1.tar.gz"
ASSET_URL = f"https://github.com/leblancfg/leblancfg.github.io/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
EXPECTED_SHA256 = "742f89eed0fa381570111b8b0fb82d97a44d40d13c14a3fcf6bfcd848a6751dd"
CACHE_DIR = Path.home() / ".cache" / "leblancfg" / RELEASE_TAG
ARCHIVE_PATH = CACHE_DIR / ASSET_NAME
EXTRACTED_DIR = CACHE_DIR / "crossfit-open-2026-compact-v1"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(tar: tarfile.TarFile, destination: Path) -> None:
    destination = destination.resolve()
    for member in tar.getmembers():
        target = (destination / member.name).resolve()
        if destination not in target.parents and target != destination:
            raise RuntimeError(f"unsafe tar member: {member.name}")
    tar.extractall(destination)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    urllib.request.urlretrieve(ASSET_URL, ARCHIVE_PATH)

actual_sha256 = sha256_file(ARCHIVE_PATH)
assert actual_sha256 == EXPECTED_SHA256, actual_sha256

if not (EXTRACTED_DIR / "metadata.json").exists():
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        safe_extract(tar, CACHE_DIR)

display(Markdown(f"Downloaded `{ASSET_NAME}` and verified SHA-256 `{actual_sha256}`."))


Downloaded `crossfit-open-2026-compact-v1.tar.gz` and verified SHA-256 `742f89eed0fa381570111b8b0fb82d97a44d40d13c14a3fcf6bfcd848a6751dd`.

In [2]:
def load_csv(name: str) -> list[dict[str, str]]:
    with (EXTRACTED_DIR / name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def int_or_none(value: str | None) -> int | None:
    return int(value) if value not in (None, "") else None

def float_or_none(value: str | None) -> float | None:
    return float(value) if value not in (None, "") else None

def fmt_int(value: int | str | None) -> str:
    if value in (None, ""):
        return ""
    return f"{int(value):,}"

def markdown_table(headers: list[str], rows: list[list[object]]) -> Markdown:
    def cell(value: object) -> str:
        return str(value).replace("|", "\|")
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(cell(value) for value in row) + " |")
    return Markdown("\n".join(lines))

metadata = json.loads((EXTRACTED_DIR / "metadata.json").read_text(encoding="utf-8"))
divisions = load_csv("divisions.csv")
athletes = load_csv("athletes.csv")
entries = load_csv("leaderboard_entries.csv")
scores = load_csv("workout_scores.csv")
benchmarks = load_csv("sample_benchmark_stats.csv")


## Reproduce the release counts

The release contains the complete normalized leaderboard crawl for every 2026 Open division. The SQLite checkpoint had raw fetch bodies too; those are deliberately absent here.


In [3]:
actual_counts = {
    "divisions.csv": len(divisions),
    "athletes.csv": len(athletes),
    "leaderboard_entries.csv": len(entries),
    "workout_scores.csv": len(scores),
    "sample_benchmark_stats.csv": len(benchmarks),
}
count_rows = []
for filename, expected in metadata["exported_row_counts"].items():
    actual = actual_counts[filename]
    count_rows.append([filename, fmt_int(expected), fmt_int(actual), "yes" if expected == actual else "no"])
display(markdown_table(["File", "Metadata rows", "Loaded rows", "Matches"], count_rows))


| File | Metadata rows | Loaded rows | Matches |
| --- | --- | --- | --- |
| divisions.csv | 23 | 23 | yes |
| athletes.csv | 254,531 | 254,531 | yes |
| leaderboard_entries.csv | 388,423 | 388,423 | yes |
| workout_scores.csv | 1,165,269 | 1,165,269 | yes |
| sample_benchmark_stats.csv | 165 | 165 | yes |

## Percentile cutoffs by sex and age-group division

Percentiles here are rank-derived. For a division with `N` athletes, the percentile for rank `r` is `100 * (1 - ((r - 1) / (N - 1)))`. The cutoff below is the worst rank still at or above the requested percentile. Because ties can share a rank, `athletes_at_or_above` can differ slightly from the rank number.


In [4]:
division_by_id = {row["division_id"]: row for row in divisions if row["category"] != "team"}
entries_by_division: dict[str, list[dict[str, str]]] = defaultdict(list)
for row in entries:
    if row["division_id"] in division_by_id:
        entries_by_division[row["division_id"]].append(row)

cutoffs = []
for division_id, division_entries in entries_by_division.items():
    division = division_by_id[division_id]
    for percentile in (50, 75, 90, 95):
        eligible = [
            row
            for row in division_entries
            if float_or_none(row["performance_percentile"]) is not None
            and float(row["performance_percentile"]) >= percentile
        ]
        cutoff_rank = max(int(row["overall_rank"]) for row in eligible if row["overall_rank"])
        cutoff_row = sorted(
            (row for row in division_entries if int_or_none(row["overall_rank"]) == cutoff_rank),
            key=lambda row: int(row["athlete_id"]) if row["athlete_id"].isdigit() else row["athlete_id"],
        )[0]
        cutoffs.append({
            "division_id": division_id,
            "division_name": division["division_name"],
            "sex": division["sex"],
            "category": division["category"],
            "total_competitors": int(division["total_competitors"]),
            "percentile": percentile,
            "athletes_at_or_above": len(eligible),
            "rank_cutoff": cutoff_rank,
            "overall_score": cutoff_row["overall_score"],
            "athlete_id": cutoff_row["athlete_id"],
        })

rank_rows = []
for division in division_by_id.values():
    division_cutoffs = {row["percentile"]: row for row in cutoffs if row["division_id"] == division["division_id"]}
    rank_rows.append([
        division["sex"],
        division["division_name"],
        fmt_int(division["total_competitors"]),
        fmt_int(division_cutoffs[50]["rank_cutoff"]),
        fmt_int(division_cutoffs[75]["rank_cutoff"]),
        fmt_int(division_cutoffs[90]["rank_cutoff"]),
        fmt_int(division_cutoffs[95]["rank_cutoff"]),
    ])
display(markdown_table(["Sex", "Division", "Athletes", "50th", "75th", "90th", "95th"], rank_rows))


| Sex | Division | Athletes | 50th | 75th | 90th | 95th |
| --- | --- | --- | --- | --- | --- | --- |
| M | Men | 127,113 | 63,557 | 31,779 | 12,712 | 6,356 |
| F | Women | 105,959 | 52,980 | 26,490 | 10,596 | 5,298 |
| M | Men 45-49 | 14,691 | 7,346 | 3,673 | 1,470 | 735 |
| F | Women 45-49 | 11,248 | 5,624 | 2,812 | 1,125 | 563 |
| M | Men 50-54 | 8,486 | 4,243 | 2,122 | 849 | 425 |
| F | Women 50-54 | 6,051 | 3,025 | 1,513 | 606 | 302 |
| M | Men 55-59 | 5,397 | 2,699 | 1,349 | 540 | 270 |
| F | Women 55-59 | 4,143 | 2,072 | 1,036 | 415 | 208 |
| M | Men 40-44 | 21,970 | 10,985 | 5,493 | 2,197 | 1,099 |
| F | Women 40-44 | 17,264 | 8,632 | 4,316 | 1,727 | 864 |
| M | Boys 14-15 | 1,077 | 539 | 270 | 108 | 53 |
| F | Girls 14-15 | 1,049 | 525 | 263 | 105 | 53 |
| M | Boys 16-17 | 1,195 | 598 | 299 | 120 | 60 |
| F | Girls 16-17 | 1,108 | 553 | 277 | 111 | 56 |
| M | Men 35-39 | 29,171 | 14,586 | 7,293 | 2,918 | 1,459 |
| F | Women 35-39 | 22,673 | 11,337 | 5,669 | 2,268 | 1,134 |
| M | Men 60-64 | 2,825 | 1,413 | 707 | 283 | 142 |
| F | Women 60-64 | 2,325 | 1,163 | 582 | 233 | 117 |
| M | Men 65-69 | 1,329 | 665 | 333 | 132 | 67 |
| F | Women 65-69 | 1,230 | 614 | 308 | 123 | 62 |
| M | Men 70+ | 740 | 369 | 185 | 74 | 36 |
| F | Women 70+ | 739 | 370 | 184 | 74 | 37 |

The rank cutoffs scale with field size, which is exactly what a rank-derived percentile should do. The more useful training view is the boundary row itself: the overall score at that rank, and the three workout scores that produced it.


In [5]:
score_by_entry: dict[tuple[str, str], dict[int, str]] = defaultdict(dict)
for row in scores:
    score_by_entry[(row["division_id"], row["athlete_id"])][int(row["workout_ordinal"])] = row["score_display"]

focus_divisions = {"Men", "Women", "Men 35-39", "Women 35-39", "Men 40-44", "Women 40-44"}
detail_rows = []
for row in cutoffs:
    if row["division_name"] not in focus_divisions:
        continue
    workout_scores = score_by_entry[(row["division_id"], row["athlete_id"])]
    detail_rows.append([
        row["division_name"],
        row["percentile"],
        fmt_int(row["rank_cutoff"]),
        fmt_int(row["athletes_at_or_above"]),
        row["overall_score"],
        workout_scores.get(1, ""),
        workout_scores.get(2, ""),
        workout_scores.get(3, ""),
    ])
display(markdown_table(["Division", "Percentile", "Rank cutoff", "Athletes at/above", "Overall score", "Workout 1", "Workout 2", "Workout 3"], detail_rows))


| Division | Percentile | Rank cutoff | Athletes at/above | Overall score | Workout 1 | Workout 2 | Workout 3 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Men | 50 | 63,557 | 63,557 | 193186 | 153 reps | 112 reps | 172 reps |
| Men | 75 | 31,779 | 31,779 | 103601 | 191 reps | 119 reps | 199 reps |
| Men | 90 | 12,712 | 12,712 | 47632 | 228 reps | 12:41 | 204 reps |
| Men | 95 | 6,356 | 6,356 | 25748 | 271 reps | 12:05 | 201 reps |
| Women | 50 | 52,980 | 52,980 | 161846 | 144 reps | 89 reps | 131 reps |
| Women | 75 | 26,490 | 26,490 | 86347 | 166 reps | 112 reps | 192 reps |
| Women | 90 | 10,596 | 10,597 | 36768 | 243 reps | 112 reps | 204 reps |
| Women | 95 | 5,298 | 5,298 | 20734 | 245 reps | 121 reps | 223 reps |
| Men 40-44 | 50 | 10,985 | 10,986 | 33445 | 184 reps | 100 reps | 138 reps |
| Men 40-44 | 75 | 5,493 | 5,494 | 17787 | 212 reps | 112 reps | 172 reps |
| Men 40-44 | 90 | 2,197 | 2,197 | 8481 | 243 reps | 112 reps | 209 reps |
| Men 40-44 | 95 | 1,099 | 1,099 | 4780 | 239 reps | 14:10 | 199 reps |
| Women 40-44 | 50 | 8,632 | 8,632 | 26349 | 152 reps | 112 reps - s | 148 reps |
| Women 40-44 | 75 | 4,316 | 4,316 | 14138 | 182 reps | 112 reps | 182 reps |
| Women 40-44 | 90 | 1,727 | 1,727 | 5867 | 228 reps | 112 reps | 195 reps |
| Women 40-44 | 95 | 864 | 865 | 3332 | 241 reps | 112 reps | 227 reps |
| Men 35-39 | 50 | 14,586 | 14,586 | 44171 | 192 reps | 91 reps | 161 reps |
| Men 35-39 | 75 | 7,293 | 7,293 | 23967 | 232 reps | 120 reps | 161 reps |
| Men 35-39 | 90 | 2,918 | 2,918 | 11119 | 230 reps | 12:55 | 202 reps |
| Men 35-39 | 95 | 1,459 | 1,460 | 6205 | 248 reps | 12:54 | 215 reps |
| Women 35-39 | 50 | 11,337 | 11,337 | 34515 | 164 reps | 74 reps | 134 reps |
| Women 35-39 | 75 | 5,669 | 5,670 | 18531 | 197 reps | 112 reps | 180 reps |
| Women 35-39 | 90 | 2,268 | 2,269 | 7903 | 235 reps | 112 reps | 226 reps |
| Women 35-39 | 95 | 1,134 | 1,134 | 4526 | 239 reps | 126 reps | 223 reps |

For the target division I care about most, Men 35-39, 90th percentile in the 2026 Open meant rank 2,918 out of 29,171, with a boundary overall score of 11,119. Moving from 90th to 95th percentile cut the allowed rank roughly in half, to 1,459, and the boundary overall score dropped to 6,205.

That is the framing I wanted from this dataset: not a vague claim about being competitive, but a reproducible cutoff table that turns sex and age group into a concrete leaderboard target.


## Benchmark profile caveats

The leaderboard tables above answer the rank question well: for a given division and percentile, they identify the boundary athlete and the Open workout scores at that boundary. The benchmark fields are weaker evidence. They come from public CrossFit athlete profiles, are self-reported, and usually represent lifetime PR values rather than current-state fitness.

In this compact release, the benchmark parser was only run as a small proof sample: **165 benchmark rows**, mostly from elite men who also appear in the Men and Men 35-39 leaderboards. That is enough to show the shape of a benchmark layer and to compute example equivalencies, but it is not enough to claim true population percentiles for every division.


In [6]:
from statistics import median

def seconds_to_mmss(seconds: float | int | None) -> str:
    if seconds in (None, ""):
        return ""
    seconds = int(round(float(seconds)))
    return f"{seconds // 60}:{seconds % 60:02d}"

def kg_to_lb(kg: float | int | None) -> float | None:
    return None if kg is None else float(kg) / 0.45359237

def safe_float(value: str | None) -> float | None:
    return float(value) if value not in (None, "") else None

athlete_by_id = {row["athlete_id"]: row for row in athletes}
benchmark_by_athlete: dict[str, dict[str, dict[str, str]]] = defaultdict(dict)
for row in benchmarks:
    benchmark_by_athlete[row["athlete_id"]][row["stat_name"]] = row

coverage_rows = []
for stat_name in sorted({row["stat_name"] for row in benchmarks}):
    rows = [row for row in benchmarks if row["stat_name"] == stat_name]
    coverage_rows.append([stat_name, len(rows), len({row["athlete_id"] for row in rows})])

display(markdown_table(["Published benchmark", "Rows", "Athletes"], coverage_rows))


| Published benchmark | Rows | Athletes |
| --- | --- | --- |
| Back Squat 1RM | 17 | 17 |
| Bench Press 1RM | 1 | 1 |
| CHAD1000x | 1 | 1 |
| Clean & Jerk 1RM | 17 | 17 |
| DT | 1 | 1 |
| Deadlift 1RM | 17 | 17 |
| Deadlift 5RM | 1 | 1 |
| Diane | 1 | 1 |
| Fight Gone Bad | 10 | 10 |
| Filthy 50 | 4 | 4 |
| Fran | 14 | 14 |
| Front Squat 1RM | 2 | 2 |
| Grace | 13 | 13 |
| Grettel | 1 | 1 |
| Havana | 1 | 1 |
| Heavy Grace | 1 | 1 |
| Helen | 6 | 6 |
| Holleyman | 1 | 1 |
| Isabel | 1 | 1 |
| Jackie | 1 | 1 |
| Karen | 1 | 1 |
| Linda | 1 | 1 |
| Power Clean 1RM | 1 | 1 |
| Power Snatch 1RM | 1 | 1 |
| Pull-ups | 10 | 10 |
| Randy | 1 | 1 |
| Row 5K | 1 | 1 |
| Run 1 Mile | 1 | 1 |
| Run 400m | 7 | 7 |
| Run 5k | 10 | 10 |
| Shoulder Press 1RM | 1 | 1 |
| Squat Clean 1RM | 1 | 1 |
| Squat Snatch 1RM | 17 | 17 |
| Thruster 1RM | 1 | 1 |

## Equivalence functions for the four benchmark families

These are not target models. They are explicit bridges from the benchmark tests CrossFit profiles actually publish to the higher-level qualities people usually ask about. Because the source is self-reported profile data, treat the values as published benchmark PRs attached to athletes above a leaderboard cutoff, not as current fitness measurements.

**Steady-state cardio: FTP-equivalent.** CrossFit profiles usually do not publish cycling FTP. The closest public profile tests in this sample are `Run 5k` and, much more rarely, `Row 5K`. For a 5k run, estimate an FTP-like cycling value in W/kg by converting 5k speed into a critical-speed proxy, then applying an economy/efficiency bridge:

`running_speed_mps = 5000 / run_5k_seconds`

`critical_speed_mps ≈ 0.92 × running_speed_mps`

`ftp_equivalent_wkg ≈ critical_speed_mps × 4.184 × 0.23`

The `4.184` term is the common running-energy-cost approximation of about 1 kcal/kg/km. The `0.23` term treats cycling FTP as external mechanical power at about 23% gross efficiency. The result is not “the athlete’s cycling FTP”; it is a steady-state engine equivalence using the published running test.

**High-energy cardio: 400m speed.** Use `Run 400m` directly as a high-power cardio benchmark. The useful value is speed, not FTP:

`speed_mps = 400 / run_400m_seconds`

It is a short glycolytic/speed-endurance test, so I would not convert it to FTP. It answers a different question: can the athlete produce high output for roughly one minute?

**Slow strength: CrossFit Total.** The closest slow-strength abstraction is CrossFit Total: back squat + shoulder press + deadlift. If a profile publishes all three components, reconstruct it directly:

`crossfit_total_kg = back_squat_1rm_kg + shoulder_press_1rm_kg + deadlift_1rm_kg`

If one component is missing, do not fill it with a model. Mark it missing.

**Fast strength: 1RM clean & jerk.** Use `Clean & Jerk 1RM` directly, shown both in kilograms and as a bodyweight multiple where body mass exists. This is the cleanest fast-strength proxy in the published profile fields because it combines speed, coordination, and maximal loading.


In [7]:
def run_5k_to_ftp_equivalent_wkg(seconds: float) -> float:
    running_speed_mps = 5000 / seconds
    critical_speed_mps = 0.92 * running_speed_mps
    return critical_speed_mps * 4.184 * 0.23

def run_400_speed_mps(seconds: float) -> float:
    return 400 / seconds

def benchmark_value_kg(stats: dict[str, dict[str, str]], name: str) -> float | None:
    row = stats.get(name)
    return safe_float(row.get("value_kg")) if row else None

def benchmark_time_seconds(stats: dict[str, dict[str, str]], name: str) -> float | None:
    row = stats.get(name)
    return safe_float(row.get("time_seconds")) if row else None

def crossfit_total_kg(stats: dict[str, dict[str, str]]) -> float | None:
    parts = [benchmark_value_kg(stats, name) for name in ("Back Squat 1RM", "Shoulder Press 1RM", "Deadlift 1RM")]
    return sum(parts) if all(part is not None for part in parts) else None

def athlete_benchmark_summary(athlete_id: str) -> dict[str, float | None]:
    stats = benchmark_by_athlete.get(athlete_id, {})
    run5k = benchmark_time_seconds(stats, "Run 5k")
    run400 = benchmark_time_seconds(stats, "Run 400m")
    clean_jerk = benchmark_value_kg(stats, "Clean & Jerk 1RM")
    cft = crossfit_total_kg(stats)
    body_mass = safe_float(athlete_by_id.get(athlete_id, {}).get("weight_kg"))
    return {
        "ftp_equiv_wkg": run_5k_to_ftp_equivalent_wkg(run5k) if run5k else None,
        "run_5k_seconds": run5k,
        "run_400_seconds": run400,
        "run_400_mps": run_400_speed_mps(run400) if run400 else None,
        "clean_jerk_kg": clean_jerk,
        "clean_jerk_bw": clean_jerk / body_mass if clean_jerk and body_mass else None,
        "crossfit_total_kg": cft,
        "crossfit_total_bw": cft / body_mass if cft and body_mass else None,
    }

sample_rows = []
for athlete_id in benchmark_by_athlete:
    summary = athlete_benchmark_summary(athlete_id)
    linked_entries = [row for row in entries if row["athlete_id"] == athlete_id and row["division_id"] in division_by_id]
    for entry in linked_entries:
        division = division_by_id[entry["division_id"]]
        sample_rows.append({
            "athlete_id": athlete_id,
            "division_name": division["division_name"],
            "performance_percentile": float_or_none(entry["performance_percentile"]),
            **summary,
        })

def median_and_n(values: list[float | None], *, digits: int = 1, formatter=None) -> str:
    present = [value for value in values if value is not None]
    if not present:
        return ""
    value = median(present)
    rendered = formatter(value) if formatter else f"{value:.{digits}f}"
    return f"{rendered} (n={len(present)})"

percentile_rows = []
for division_name in ("Men", "Men 35-39"):
    for threshold in (90, 95, 99):
        rows = [row for row in sample_rows if row["division_name"] == division_name and row["performance_percentile"] is not None and row["performance_percentile"] >= threshold]
        percentile_rows.append([
            division_name,
            f">= {threshold}th",
            len(rows),
            median_and_n([row["ftp_equiv_wkg"] for row in rows], digits=2),
            median_and_n([row["run_400_seconds"] for row in rows], formatter=seconds_to_mmss),
            median_and_n([row["clean_jerk_kg"] for row in rows], digits=0),
            median_and_n([row["clean_jerk_bw"] for row in rows], digits=2),
            median_and_n([row["crossfit_total_kg"] for row in rows], digits=0),
            median_and_n([row["crossfit_total_bw"] for row in rows], digits=2),
        ])

display(markdown_table([
    "Division", "Leaderboard filter", "Profile athletes", "FTP-equiv W/kg", "400m", "C&J kg", "C&J xBW", "CFT kg", "CFT xBW"
], percentile_rows))


| Division | Leaderboard filter | Profile athletes | FTP-equiv W/kg | 400m | C&J kg | C&J xBW | CFT kg | CFT xBW |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Men | >= 90th | 17 | 3.92 (n=16) | 0:59 (n=16) | 156 (n=17) | 1.68 (n=17) | 599 (n=15) | 6.22 (n=15) |
| Men | >= 95th | 17 | 3.92 (n=16) | 0:59 (n=16) | 156 (n=17) | 1.68 (n=17) | 599 (n=15) | 6.22 (n=15) |
| Men | >= 99th | 17 | 3.92 (n=16) | 0:59 (n=16) | 156 (n=17) | 1.68 (n=17) | 599 (n=15) | 6.22 (n=15) |
| Men 35-39 | >= 90th | 17 | 3.92 (n=16) | 0:59 (n=16) | 156 (n=17) | 1.68 (n=17) | 599 (n=15) | 6.22 (n=15) |
| Men 35-39 | >= 95th | 17 | 3.92 (n=16) | 0:59 (n=16) | 156 (n=17) | 1.68 (n=17) | 599 (n=15) | 6.22 (n=15) |
| Men 35-39 | >= 99th | 17 | 3.92 (n=16) | 0:59 (n=16) | 156 (n=17) | 1.68 (n=17) | 599 (n=15) | 6.22 (n=15) |

The repeated rows are a data limitation, not a conclusion that the 90th, 95th, and 99th percentile athletes have identical benchmark profiles. The compact release only includes a small benchmark proof sample, and that sample is concentrated among athletes who are above all three cutoffs.

The useful thing in this pass is the shape of the benchmark layer: use published profile PRs, expose missingness per metric, and avoid filling gaps with generic modelled values. The next proper dataset pass should crawl public profiles around cutoff ranks for each sex/division/percentile band. Then this table can become a real benchmark-percentile table with `n`, median, 25th/75th, and missingness rate per metric.
